# Assignment No. 8
## Text Generation: Implement a Long Short-Term Memory (LSTM) network using TensorFlow 2
**Student Name:** Aditya Chandrajit Deore  
**PRN:** 124B1F116  
**Course:** Generative AI (GEN AI)  


### Part A: Load and Understand Data

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.21.0


In [2]:
text = """
Generative AI can create new text, images and music.
Language models learn the patterns of words from large amounts of text.
An LSTM network can remember important information from earlier words.
Text generation predicts the next word in a sentence.
Deep learning models are trained using large datasets.
Neural networks learn patterns by adjusting their weights.
Data preprocessing makes the dataset clean and ready for training.
A tokenizer converts words into numbers for the model.
The embedding layer turns each word into a dense vector.
The LSTM layer learns the order of words in a sentence.
The dense layer predicts the probability of every word.
Training a model needs many epochs to reduce the loss.
A good model can generate new sentences from a seed text.
Machine learning helps computers learn from data.
Artificial intelligence is changing education and healthcare.
Recurrent neural networks are used for sequence data.
The forget gate decides which information the LSTM should remove.
The input gate decides which new information the LSTM should store.
The output gate decides what the LSTM should pass to the next step.
Generative models can create content that looks like real data.
Students use Python and TensorFlow to build deep learning models.
A seed text is the starting point for text generation.
The model predicts one word at a time to build a sentence.
Cleaning the text improves the quality of the training data.
Good training data helps the model generate meaningful text.
"""

# Store each line as a row in a DataFrame
lines = [l.strip() for l in text.split('\n') if l.strip() != ""]
df = pd.DataFrame({'sentence': lines})

print("Number of sentences:", len(df))
print("Number of words:", sum(len(s.split()) for s in lines))
df.head(10)


Number of sentences: 25
Number of words: 218


,sentence
0,"Generative AI can create new text, images and music."
1,Language models learn the patterns of words from large amounts of text.
2,An LSTM network can remember important information from earlier words.
3,Text generation predicts the next word in a sentence.
4,Deep learning models are trained using large datasets.
5,Neural networks learn patterns by adjusting their weights.
6,Data preprocessing makes the dataset clean and ready for training.
7,A tokenizer converts words into numbers for the model.
8,The embedding layer turns each word into a dense vector.
9,The LSTM layer learns the order of words in a sentence.


### Part B: Preprocess Text

In [3]:
# 1. Clean and normalize: Lowercase and remove punctuation
df['clean'] = (df['sentence'].str.lower()
               .str.replace(r'[^a-z\s]', '', regex=True)
               .str.strip())
df[['sentence', 'clean']].head(5)


,sentence,clean
0,"Generative AI can create new text, images and ...",generative ai can create new text images and m...
1,Language models learn the patterns of words fr...,language models learn the patterns of words fr...
2,An LSTM network can remember important informa...,an lstm network can remember important informa...
3,Text generation predicts the next word in a se...,text generation predicts the next word in a se...
4,Deep learning models are trained using large d...,deep learning models are trained using large d...


In [4]:
# 2. Tokenize with the Keras Tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['clean'])

total_words = len(tokenizer.word_index) + 1  # +1 because index 0 is reserved for padding
print("Vocabulary size:", total_words)
print("First 15 entries of word index:")
print(dict(list(tokenizer.word_index.items())[:15]))


Vocabulary size: 122
First 15 entries of word index:
{'the': 1, 'a': 2, 'text': 3, 'data': 4, 'of': 5, 'lstm': 6, 'model': 7, 'can': 8, 'and': 9, 'models': 10, 'words': 11, 'from': 12, 'word': 13, 'for': 14, 'training': 15}


In [5]:
# 3. Convert words into numerical sequences
sample = df['clean'][0]
print(sample)
print(tokenizer.texts_to_sequences([sample])[0])


generative ai can create new text images and music
[27, 47, 8, 28, 17, 3, 48, 9, 49]


### Part C: Create Sequences

In [6]:
# 1. Divide each sentence into growing sequences (n-grams)
input_sequences = []

for line in df['clean']:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(2, len(token_list) + 1):
        input_sequences.append(token_list[:i])

print("Total sequences:", len(input_sequences))
print("Example sequences:", input_sequences[:3])


Total sequences: 218
Example sequences: [[27, 47], [27, 47, 8], [27, 47, 8, 28]]


In [7]:
# 3. Apply padding so all sequences have the same length
max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

print("Maximum sequence length:", max_len)
print("Padded shape:", input_sequences.shape)


Maximum sequence length: 13
Padded shape: (218, 13)


In [8]:
# 2. Input (X) = all words except the last, target (y) = the last word
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (218, 12)
y shape: (218,)


### Part D: Build the LSTM Model

In [9]:
model = Sequential([
    Input(shape=(max_len - 1,)),
    Embedding(input_dim=total_words, output_dim=64),
    LSTM(128),
    Dense(total_words, activation='softmax')
])

# Integer targets, so sparse categorical cross-entropy is used (no one-hot needed)
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


Model: "sequential"
┌─────────────────────────────────┬────────────────────────┬───────────────┐
│ Layer (type)                    │ Output Shape           │       Param # │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 12, 64)         │         7,808 │
│ lstm (LSTM)                     │ (None, 128)            │        98,816 │
│ dense (Dense)                   │ (None, 122)            │        15,738 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 122,362 (477.98 KB)
 Trainable params: 122,362 (477.98 KB)
 Non-trainable params: 0 (0.00 B)


### Part E: Train the Model

In [10]:
history = model.fit(X, y, epochs=200, batch_size=32, verbose=1)


Epoch 1/200
7/7 [==============================] - 1s 29ms/step - accuracy: 0.0183 - loss: 4.8040
Epoch 10/200
7/7 [==============================] - 0s 35ms/step - accuracy: 0.0734 - loss: 4.3533
Epoch 50/200
7/7 [==============================] - 0s 30ms/step - accuracy: 0.2844 - loss: 2.8540
Epoch 100/200
7/7 [==============================] - 0s 28ms/step - accuracy: 0.7431 - loss: 1.1524
Epoch 150/200
7/7 [==============================] - 0s 28ms/step - accuracy: 0.9404 - loss: 0.3842
Epoch 200/200
7/7 [==============================] - 0s 29ms/step - accuracy: 0.9771 - loss: 0.1654


### Part F: Generate Text

In [14]:
index_to_word = {index: word for word, index in tokenizer.word_index.items()}

def generate_text(seed_text, next_words, temperature=0.7):
    for _ in range(next_words):
        # Convert the current text to a padded sequence
        tokens = tokenizer.texts_to_sequences([seed_text.lower()])[0]
        tokens = pad_sequences([tokens], maxlen=max_len - 1, padding='pre')
        
        # Predict probabilities of the next word
        probs = model.predict(tokens, verbose=0)[0].astype('float64')
        
        # Apply temperature and pick the next word
        probs = np.log(probs + 1e-8) / temperature
        probs = np.exp(probs) / np.sum(np.exp(probs))
        next_index = np.random.choice(len(probs), p=probs)
        
        seed_text += " " + index_to_word.get(next_index, "")
    return seed_text


In [15]:
print(generate_text("generative ai", 8))
print(generate_text("the lstm layer", 8))
print(generate_text("a good model", 8, temperature=1.0))
print(generate_text("text generation", 8, temperature=0.5))


generative ai can create new text images and music music
the lstm layer learns the order of words in a sentence
a good model can generate new sentences from a seed text
text generation predicts the next word in a sentence sentence
